Decision Stump

In [4]:
import numpy as np

In [14]:
class DecisionStump:
    def __init__(self):
        self.feature_idx=None
        self.threshold=None
        self.polarity=1
        self.alpha=None
    def fit(self,X,y,weights):
        n_samples,n_features=X.shape
        best_error=float("inf")

        for f in range(n_features):
            thresholds = np.unique(X[:, f])
            for thresh in thresholds:
                for polarity in [1, -1]:
                    pred = np.ones(n_samples)
                    pred[polarity * X[:, f] < polarity * thresh] = -1
                    error = np.sum(weights[pred != y])
                    if error < best_error:
                        best_error = error
                        self.feature_idx = f
                        self.threshold = thresh
                        self.polarity = polarity
    def predict(self, X):
        n = X.shape[0]
        pred = np.ones(n)
        idx = self.polarity * X[:, self.feature_idx] < self.polarity * self.threshold
        pred[idx] = -1
        return pred

In [6]:
X = np.array([[1],[2],[3],[4]])
y = np.array([-1,-1, 1, 1])
w = np.ones(4) / 4

In [7]:
X = np.array([[1],[2],[3],[4]])
y = np.array([1, 1,-1,-1])
w = np.ones(4) / 4

Adaboost From scratch

In [15]:
class AdaBoostScratch:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.stumps = []
        self.alphas = []

    def fit(self, X, y):
        n = X.shape[0]
        weights = np.full(n, 1 / n)

        for _ in range(self.n_estimators):
            stump = DecisionStump()
            stump.fit(X, y, weights)
            pred = stump.predict(X)

            err = np.sum(weights[pred != y])
            err = np.clip(err, 1e-10, 1 - 1e-10)

            alpha = 0.5 * np.log((1 - err) / err)
            weights *= np.exp(-alpha * y * pred)
            weights /= weights.sum()

            stump.alpha = alpha
            self.stumps.append(stump)
            self.alphas.append(alpha)

    def predict(self, X):
        total = sum(a * s.predict(X) for a, s in zip(self.alphas, self.stumps))
        return np.sign(total)

In [16]:
X = np.array([[1],[2],[3],[4]], dtype=float)
y = np.array([-1,-1, 1, 1], dtype=float)
model = AdaBoostScratch(n_estimators=5)
model.fit(X, y)
preds = model.predict(X)

In [17]:
preds

array([-1., -1.,  1.,  1.])